# Üç komşu yönteminde ulaşılabilir en düşük hata
Fotoğraflı modelin üçlüsü ile laboratuvar eğrisi bilindiğinde seçilebilecek en iyi üçlüyü karşılaştırıyoruz.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.preprocessing import StandardScaler

root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path("work/soil-data")
data = next(root.rglob("Training_labels_updated.csv")).parent
labels = pd.read_csv(data / "Training_labels_updated.csv").set_index("sample_id")
cameras = pd.read_csv(data / "ppm_updated.csv")
photos = sorted(data.joinpath("Training-All_Photos_updated").rglob("*.jpg"))
print(f"{len(labels)} numune, {len(photos)} eğitim fotoğrafı")

24 numune, 127 eğitim fotoğrafı


## 1. Fotoğrafı bir avuç sayıyla anlatmak
Bütün fotoğrafların merkezinden yaklaşık 100 × 100 mm alan alıyoruz. Kamera tablosundaki piksel/mm değerini gerçek dosyanın çözünürlüğüne uyarlıyoruz. Bu kalibrasyon, çekim uzaklığının kamera tablosunda varsayılana yakın olduğunu kabul eder.

Bu alanı griye çevirip 400 × 400 piksele getiriyoruz. Yatay ve dikey yönde, birbirinden 0,25 mm ile 8 mm arasında uzakta olan noktaların ne kadar farklı aydınlıkta göründüğünü ölçüyoruz. Böylece her fotoğrafta aynı altı fiziksel uzaklığı karşılaştırıyoruz. Her numunenin fotoğraflarını, her ölçünün ortanca değerini alarak birleştiriyoruz.

In [2]:
def physical_patch(path, camera):
    with Image.open(path) as image:
        image = ImageOps.exif_transpose(image)
        w, h = image.size
        pixels_per_mm = camera.ppm * max(w, h) / max(camera.width, camera.height)
        side = round(100 * pixels_per_mm)
        assert 0 < side <= min(w, h)
        x, y = (w - side) // 2, (h - side) // 2
        return image.crop((x, y, x + side, y + side)).convert("RGB").resize((400, 400), Image.Resampling.LANCZOS)

def texture(patch):
    gray = np.asarray(patch.convert("L"), dtype=float)
    low, high = np.percentile(gray, [5, 95])
    gray = np.clip((gray - low) / max(high - low, 1.0), 0, 1)
    values = []
    for offset in [1, 2, 4, 8, 16, 32]:
        horizontal = np.abs(gray[:, offset:] - gray[:, :-offset]).mean()
        vertical = np.abs(gray[offset:, :] - gray[:-offset, :]).mean()
        values.append((horizontal + vertical) / 2)
    return values

camera_by_name = {c.phone.replace(" ", "_").lower(): c for c in cameras.itertuples()}
rows = []
examples = {}
for path in photos:
    phone, sample, number = path.stem.rsplit("_", 2)
    patch = physical_patch(path, camera_by_name[phone.lower()])
    rows.append([sample, *texture(patch)])
    examples.setdefault(sample, patch)
features = pd.DataFrame(rows, columns=["sample_id", "0.25", "0.5", "1", "2", "4", "8"])
features = features.groupby("sample_id").median()
assert set(features.index) == set(labels.index)
features = features.loc[labels.index]

## 2. Gerçek cevabı saklayıp tahmin yapmak
Her turda yalnızca geride kalan 23 numuneden ölçekleme değerlerini öğreniyoruz. Altı doku ölçüsüne göre uzaklık hesaplayıp en yakın üç komşuyu seçiyoruz. Komşuların ölçümlerinin ortalaması tahminimiz oluyor. Karşılaştırma için aynı 23 numunenin tamamının ortalamasını da hesaplıyoruz.

In [3]:
sizes = labels.columns.astype(float).to_numpy()
widths = np.diff(np.log10(sizes))

def emd(observed, predicted):
    return np.abs(observed[:-1] - predicted[:-1]) @ widths

scores, neighbors, predictions, averages = [], {}, {}, {}
for sample in labels.index:
    train = features.drop(index=sample)
    scaler = StandardScaler().fit(train)
    distance = np.linalg.norm(scaler.transform(train) - scaler.transform(features.loc[[sample]]), axis=1)
    nearest = train.index[np.argsort(distance, kind="stable")[:3]].tolist()
    predicted = labels.loc[nearest].mean().to_numpy()
    baseline = labels.loc[train.index].mean().to_numpy()
    observed = labels.loc[sample].to_numpy()
    assert sample not in nearest and len(set(nearest)) == 3
    assert np.all(np.diff(predicted) >= -1e-10) and np.isclose(predicted[-1], 100)
    scores.append((sample, emd(observed, baseline), emd(observed, predicted)))
    neighbors[sample], predictions[sample], averages[sample] = nearest, predicted, baseline
scores = pd.DataFrame(scores, columns=["Numune", "Ortalama", "Üç komşu"]).set_index("Numune")
print(scores.mean().round(2).to_string())
print(f"Üç komşunun daha az hata yaptığı numune sayısı: {(scores['Üç komşu'] < scores['Ortalama']).sum()}/{len(scores)}")

Ortalama    89.89
Üç komşu    44.74
Üç komşunun daha az hata yaptığı numune sayısı: 18/24


## Her saklı numune için 1.771 üçlü

Kalan 23 numuneden üç farklı numune seçip eğrilerinin eşit ağırlıklı ortalamasını alıyoruz. Aynı üçlü bütün çaplarda kullanılıyor. Laboratuvar sonucuna bakıp EMD'yi en aza indiren üçlüyü seçmek, yalnızca bu yönteme ve bu referans numunelerine özgü matematiksel bir alt sınır verir. Gerçek test numuneleri için bu seçim yapılamaz.

In [4]:
from itertools import combinations
from math import comb

records = []
for sample in labels.index:
    pool = labels.drop(index=sample)
    groups = np.array(list(combinations(range(len(pool)), 3)))
    assert len(groups) == comb(23, 3) == 1771
    candidates = pool.to_numpy()[groups].mean(axis=1)
    observed = labels.loc[sample].to_numpy()
    errors = np.abs(candidates[:, :-1] - observed[:-1]) @ widths
    best = int(errors.argmin())
    chosen = neighbors[sample]
    chosen_indices = sorted(pool.index.get_loc(n) for n in chosen)
    match = np.flatnonzero((groups == chosen_indices).all(axis=1))
    assert len(match) == 1 and np.isclose(errors[match[0]], scores.loc[sample, "Üç komşu"])
    floor = errors[best]
    model = scores.loc[sample, "Üç komşu"]
    ids = pool.index[groups[best]].tolist()
    assert sample not in ids and floor <= model + 1e-10
    assert np.isclose(floor, emd(observed, labels.loc[ids].mean().to_numpy()))
    records.append([sample, model, floor, model - floor, ", ".join(chosen), ", ".join(ids)])

table = pd.DataFrame(records, columns=["Numune", "Fotoğraf EMD", "Alt sınır", "Fark", "Fotoğrafın seçimi", "En iyi üçlü"]).set_index("Numune")
assert len(table) == 24 and round(table["Fotoğraf EMD"].mean(), 2) == 44.74
print(table.sort_values("Fark", ascending=False).round(2).to_string())
print("Ortalama:", table.iloc[:, :3].mean().round(3).to_dict())
print("Ortanca:", table.iloc[:, :3].median().round(3).to_dict())

        Fotoğraf EMD  Alt sınır    Fark Fotoğrafın seçimi       En iyi üçlü
Numune                                                                     
H405          133.53      15.28  118.25  H617, H037, H038  F827, H126, H549
H374          128.87      12.52  116.36  H030, H666, H038  H038, H368, H615
H038           93.28       5.26   88.02  H037, H617, H030  H037, H374, H637
H617           71.29       4.16   67.13  H037, H038, H405  G190, H493, H616
H516           90.35      29.99   60.36  H616, H366, H668  H181, H183, H616
H037           60.27       8.46   51.81  H617, H038, H405  H374, H516, H549
F827           63.98      19.07   44.91  G190, H617, H493  H405, H549, H668
H616           85.30      43.47   41.83  H668, H549, H126  H030, H516, H617
H030           44.43       5.18   39.24  G190, H038, F827  H405, H616, H617
H637           40.53      10.61   29.93  H183, H181, H367  H037, H038, H374
H668           33.70       4.65   29.05  H549, H126, H616  F827, G190, H549
H183        